# 03 iPhone Models Web Scraping

Scrape iPhone generation metadata from Wikipedia, normalize split generation names, and save the result as a CSV file for the analysis notebook.

In [1]:
import re
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup

## Scrape iPhone Generations

In [2]:
from io import StringIO

url = "https://en.wikipedia.org/wiki/IPhone"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) TechShop_UniProject_Scraper/1.0"
}

response = requests.get(url, headers=headers, timeout=30)
response.raise_for_status()
print(f"Downloaded iPhone reference page: HTTP {response.status_code}")

tables = pd.read_html(StringIO(response.text))

# Table containing all iPhone models and their support history
iphone_models_pd = tables[3].copy()

# Flatten the multi-level column names
iphone_models_pd.columns = [
    "_".join(str(level) for level in col if str(level) != "nan")
    if isinstance(col, tuple)
    else str(col)
    for col in iphone_models_pd.columns
]

Downloaded iPhone reference page: HTTP 200


## Normalize Split Model Names

In [3]:
iphone_models_pd = iphone_models_pd.rename(columns={
    "Model_Model_Model": "model_name",
    "Release(d)_With OS_With OS": "initial_os",
    "Release(d)_Date_Date": "release_date",
    "Discontinued_Discontinued_Discontinued": "discontinued_date",
    "Support_Ended_Ended": "support_ended",
    "Support_Final OS[a]_Final OS[a]": "final_os",
    "Support_Lifespan[b]_Max[c]": "support_lifespan_max",
    "Support_Lifespan[b]_Min[d]": "support_lifespan_min",
    "Status_Status_Status": "support_status",
})

text_columns = iphone_models_pd.select_dtypes(include="object").columns

iphone_models_pd[text_columns] = (
    iphone_models_pd[text_columns]
    .replace("\xa0", " ", regex=True)
)

iphone_models_pd["release_date"] = (
    iphone_models_pd["release_date"]
    .astype(str)
    .str.extract(r"([A-Za-z]+ \d{1,2}, \d{4})", expand=False)
)

iphone_models_pd["release_date"] = pd.to_datetime(
    iphone_models_pd["release_date"],
    errors="coerce"
)

iphone_models_pd["release_year"] = iphone_models_pd["release_date"].dt.year

for column in ["discontinued_date", "support_ended"]:
    iphone_models_pd[column] = (
        iphone_models_pd[column]
        .astype(str)
        .str.extract(r"([A-Za-z]+ \d{1,2}, \d{4})", expand=False)
    )

    iphone_models_pd[column] = (
        pd.to_datetime(iphone_models_pd[column], errors="coerce")
        .dt.strftime("%Y-%m-%d")
    )

iphone_models_pd["release_date"] = (
    pd.to_datetime(iphone_models_pd["release_date"], errors="coerce")
      .dt.strftime("%Y-%m-%d")
)

iphone_models_pd = iphone_models_pd.dropna(subset=["model_name", "release_year"])
iphone_models_pd["release_year"] = iphone_models_pd["release_year"].astype(int)

iphone_models_pd["model_name"] = (
    iphone_models_pd["model_name"]
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

expanded_rows = []

for _, row in iphone_models_pd.iterrows():
    model_name = row["model_name"]
    parts = [part.strip() for part in re.split(r"\s*/\s*", model_name)]

    for part in parts:
        expanded_row = row.copy()
        expanded_row["model_name"] = part if part.lower().startswith("iphone") else f"iPhone {part}"
        expanded_rows.append(expanded_row)

iphone_models_pd = pd.DataFrame(expanded_rows)
iphone_models_pd = iphone_models_pd.drop_duplicates(subset=["model_name", "release_year"])

print(f"iPhone generations scraped: {len(iphone_models_pd)}")
display(iphone_models_pd.head(10))

iPhone generations scraped: 52


,model_name,initial_os,release_date,discontinued_date,support_ended,final_os,support_lifespan_max,support_lifespan_min,support_status,release_year
0,iPhone,iPhone OS 1.0,2007-06-29,2008-06-09,2010-06-21,iPhone OS 3.1.3,"2 years, 11 months",2 years,Discontinued and unsupported,2007
1,iPhone 3G,iPhone OS 2.0,2008-07-11,2010-08-09,2011-03-03,iOS 4.2.1,"2 years, 7 months",6 months,Discontinued and unsupported,2008
2,iPhone 3GS,iPhone OS 3.0,2009-06-19,2012-09-12,2013-09-18,iOS 6.1.3 (6.1.6),"4 years, 2 months",1 year,Discontinued and unsupported,2009
3,iPhone 4,iOS 4.0,2010-06-24,2013-09-10,2014-09-17,iOS 7.1.2,"4 years, 2 months",1 year,Discontinued and unsupported,2010
4,iPhone 4s,iOS 5.0,2011-10-14,2014-09-09,2016-09-13,iOS 9.3.5 (9.3.6),"4 years, 10 months",2 years,Discontinued and unsupported,2011
5,iPhone 5,iOS 6.0,2012-09-21,2013-09-10,2017-09-19,iOS 10.3.3 (10.3.4),"4 years, 11 months",4 years,Discontinued and unsupported,2012
6,iPhone 5c,iOS 7.0,2013-09-20,2015-09-09,2017-09-19,iOS 10.3.3,"3 years, 11 months",2 years,Discontinued and unsupported,2013
7,iPhone 5s,iOS 7.0,2013-09-20,2016-03-21,2019-09-18,iOS 12.4.1 (12.5.8),"5 years, 11 months","3 years, 5 months",Discontinued and unsupported,2013
8,iPhone 6,iOS 8.0,2014-09-19,2016-09-07,2019-09-18,iOS 12.4.1 (12.5.8),"4 years, 11 months",3 years,Discontinued and unsupported,2014
8,iPhone 6 Plus,iOS 8.0,2014-09-19,2016-09-07,2019-09-18,iOS 12.4.1 (12.5.8),"4 years, 11 months",3 years,Discontinued and unsupported,2014


## Save iPhone Model Data

In [4]:
output_file = Path("data/wikipedia-iphone-models.csv")
iphone_models_pd.to_csv(output_file, index=False)

print(f"iPhone model data saved to {output_file}")

iPhone model data saved to data/wikipedia-iphone-models.csv
